# Choix collectif et théorie du vote — 7 méthodes de gouvernance multi-agents

**Notebook pédagogique CoursIA** · Track CI-2 (#1458, Epic #1448) · `Claude Code @ myia-po-2025:2025-Epita-Intelligence-Symbolique`.

Ce notebook illustre **l'axe gouvernance** (7 méthodes de vote + théorie du choix social formel)
vendoré dans l'essence CoursIA mais jusqu'ici non enseigné par un notebook. Il tourne **firsthand**
(cellules exécutées, sorties réelles) contre le code local
`argumentation_analysis.agents.core.governance`.

> **Privacy.** Le scénario est entièrement **synthétique et domaine-public** : un club de robotique
> choisit le menu de son repas de fin d'année. Aucun texte du corpus chiffré, aucune donnée nominative,
> aucune politique réelle. Seuls des prénoms fictifs et des plats neutres apparaissent.

## Le fil rouge : la méthode de vote change le gagnant

Marquis de Condorcet (1785) a montré qu'aucune méthode de vote n'est neutre face au choix collectif :
**deux méthodes honnêtes, appliquées aux mêmes préférences, peuvent élire deux gagnants différents** —
et le scrutin majoritaire à un tour peut élire le candidat qu'une majorité classe *dernier*
(le « perdant de Condorcet »). Le théorème d'impossibilité d'Arrow (1951) généralise :
*aucune* méthode agrégeant des préférences individuelles ne peut simultanément satisfaire quelques
exigences raisonnables (universalité, non-dictature, monotonicité, indépendance aux alternatives
non pertinentes). D'où l'importance, en IA multi-agents, de **choisir explicitement** la méthode
de gouvernance plutôt que de la présupposer.

In [1]:
# Imports + configuration.
# On importe l'essence de gouvernance depuis le code local (pin essence CoursIA à rafraîchir).
import logging
import os
import sys

logging.disable(logging.INFO)   # étouffer le bruit de logs du package, sorties propres

# Localiser la racine du dépôt (contient argumentation_analysis/) en remontant depuis le CWD.
# Le notebook peut s'exécuter depuis son propre dossier ; on remonte jusqu'au package.
_cwd = os.getcwd()
while _cwd and not os.path.isdir(os.path.join(_cwd, "argumentation_analysis")):
    _parent = os.path.dirname(_cwd)
    if _parent == _cwd:
        break
    _cwd = _parent
if os.path.isdir(os.path.join(_cwd, "argumentation_analysis")):
    sys.path.insert(0, _cwd)

import numpy as np

from argumentation_analysis.agents.core.governance.governance_agent import Agent
from argumentation_analysis.agents.core.governance import governance_methods as gm
from argumentation_analysis.agents.core.governance import social_choice as sc

np.random.seed(42)   # reproductibilité des méthodes stochastiques (byzantin, raft)
print("Essence de gouvernance chargée.")
print("7 méthodes disponibles :", ", ".join(sorted(gm.GOVERNANCE_METHODS)))

Essence de gouvernance chargée.
7 méthodes disponibles : borda, byzantine, condorcet, majority, plurality, quadratic, raft


## §1 — Le scénario : le repas de fin d'année du club de robotique

Sept membres du club doivent choisir **un** menu parmi quatre. Chacun a un ordre de préférence
strict et une *personnalité* de vote :

- **stubborn** (têtu) : vote toujours pour son premier choix ;
- **flexible** : peut s'aligner sur une majorité indicative, sinon vote pour son premier choix.

| Membre | Personnalité | Ordre de préférence (1er → dernier) |
|--------|--------------|--------------------------------------|
| Alice, Bob, Chloé | stubborn | Pizza > Burger > Sushi > Raclette |
| David, Emma | flexible | Raclette > Sushi > Burger > Pizza |
| Farid | flexible | Sushi > Burger > Raclette > Pizza |
| Gérald | flexible | Burger > Sushi > Raclette > Pizza |

Trois membres placent la **Pizza** en tête (majorité des premières places), mais une majorité
de membres (4 sur 7) classe la Pizza *dernière*. Qui devrait gagner ? Dépend de la méthode.

In [2]:
# Profil de préférences (synthétique, domaine-public).
OPTIONS = ["Pizza", "Burger", "Sushi", "Raclette"]

PROFIL = [
    ("Alice",  "stubborn", ["Pizza",   "Burger", "Sushi",   "Raclette"]),
    ("Bob",    "stubborn", ["Pizza",   "Burger", "Sushi",   "Raclette"]),
    ("Chloé",  "stubborn", ["Pizza",   "Burger", "Sushi",   "Raclette"]),
    ("David",  "flexible", ["Raclette", "Sushi", "Burger",  "Pizza"]),
    ("Emma",   "flexible", ["Raclette", "Sushi", "Burger",  "Pizza"]),
    ("Farid",  "flexible", ["Sushi",   "Burger", "Raclette","Pizza"]),
    ("Gérald", "flexible", ["Burger",  "Sushi",  "Raclette","Pizza"]),
]

agents = [Agent(nom, perso, prefs) for nom, perso, prefs in PROFIL]
ballots = [prefs for _, _, prefs in PROFIL]   # profils nus pour la théorie du choix social
print(f"{len(agents)} électeurs, {len(OPTIONS)} options : {OPTIONS}")

7 électeurs, 4 options : ['Pizza', 'Burger', 'Sushi', 'Raclette']


## §2 — Les 7 méthodes de gouvernance

Chaque méthode partage la signature `(agents, options, context) -> gagnant`. On les fait tourner
toutes sur le même profil et on compare les gagnants.

| Méthode | Principe |
|---------|----------|
| **majority / plurality** | un tour, le plus de premières places l'emporte |
| **borda** | score pondéré (n-1, n-2, …, 0) sur tout le classement |
| **condorcet** | duels pairwise ; gagnant qui bat tout le monde, sinon Borda |
| **quadratic** | budget de voix quadratique, allocation stratégique |
| **byzantine** | tolérance à des agents défaillants (votes aléatoires) |
| **raft** | élection d'un leader + acceptation majoritaire |

In [3]:
# On exécute les 7 méthodes sur le même profil.
context = {}
np.random.seed(42)

resultats = {}
for nom, fn in gm.GOVERNANCE_METHODS.items():
    resultats[nom] = fn(agents, OPTIONS, context)

print(f"{'Méthode':<14} {'Gagnant':<10}")
print("-" * 26)
for nom, gagnant in resultats.items():
    print(f"{nom:<14} {gagnant:<10}")

gagnants_uniques = set(resultats.values())
print(f"\n{len(gagnants_uniques)} gagnant(s) distinct(s) selon la méthode : {sorted(gagnants_uniques)}")

Méthode        Gagnant   
--------------------------
majority       Pizza     
plurality      Pizza     
borda          Burger    
condorcet      Burger    
quadratic      Pizza     
byzantine      Pizza     
raft           Pizza     

2 gagnant(s) distinct(s) selon la méthode : ['Burger', 'Pizza']


### Lecture du résultat

La **majorité à un tour** élit la **Pizza** (3 premières places sur 7), alors que la **Borda**
et le **Condorcet** élisent le **Burger**. Regardons pourquoi en détail.

In [4]:
# Détail Borda : score de chaque option.
n = len(OPTIONS)
scores = {o: 0 for o in OPTIONS}
for _, _, prefs in PROFIL:
    for i, o in enumerate(prefs):
        scores[o] += n - i - 1

print("Scores de Borda (top = n-1 = 3, dernier = 0) :")
for o, s in sorted(scores.items(), key=lambda kv: -kv[1]):
    barre = "█" * s
    print(f"  {o:<9} {s:>2}  {barre}")
print(f"\n→ Gagnant Borda : {max(scores, key=scores.get)}")

Scores de Borda (top = n-1 = 3, dernier = 0) :
  Burger    13  █████████████
  Sushi     12  ████████████
  Pizza      9  █████████
  Raclette   8  ████████

→ Gagnant Borda : Burger


In [5]:
# Matrice des duels pairwise → gagnant de Condorcet.
print("Duels pairwise (ligne bat colonne) : votes en faveur de la ligne\n")
header = "        " + "  ".join(f"{o[:3]:>5}" for o in OPTIONS)
print(header)
for o1 in OPTIONS:
    cellules = []
    for o2 in OPTIONS:
        if o1 == o2:
            cellules.append("   —")
        else:
            pour = sum(1 for _, _, p in PROFIL if p.index(o1) < p.index(o2))
            cellules.append(f"{pour:>5}")
    print(f"{o1[:7]:<8}" + "  ".join(cellules))

gagnant_condorcet = sc.condorcet_winner(ballots, OPTIONS)
print(f"\nGagnant de Condorcet : {gagnant_condorcet}")
print("(bat strictement chaque autre option en duel)")

Duels pairwise (ligne bat colonne) : votes en faveur de la ligne

          Piz    Bur    Sus    Rac
Pizza      —      3      3      3
Burger      4     —      4      5
Sushi       4      3     —      5
Raclett     4      2      2     —

Gagnant de Condorcet : Burger
(bat strictement chaque autre option en duel)


**Leçon n°1.** La Pizza gagne au scrutin majoritaire mais **perd tous ses duels pairwise**
(3-4 contre Burger, Sushi *et* Raclette) : c'est le **perdant de Condorcet**. À l'inverse, le
Burger bat tout le monde — c'est le **gagnant de Condorcet** et le gagnant de Borda. La majorité
à un tour a ici élu le candidat qu'une majorité *rejette*. Le choix de la méthode n'est pas un
détail technique : il décide *qui* l'emporte.

## §3 — Théorie du choix social formel

Au-delà des méthodes agentiques, le package fournit les procédures classiques du choix social
computational (qui travaillent sur des *profils de préférences* plutôt que sur des objets
`Agent`) : **approval voting**, **STV** (vote unique transférable), **Copeland**, **Schulze**.
Elles s'appliquent aux mêmes bulletins (`ballots`).

In [6]:
# Approval voting : chaque électeur approuve ses 2 premiers choix.
g_approval, comptes_approval = sc.approval_voting(ballots, OPTIONS, approval_threshold=2)
print("Approval voting (chacun approuve son top-2) :")
for o, c in sorted(comptes_approval.items(), key=lambda kv: -kv[1]):
    print(f"  {o:<9} {c} approbations")
print(f"→ Gagnant : {g_approval}")

# STV (instant runoff) — 1 siège.
gagnants_stv, tours = sc.stv(ballots, OPTIONS, seats=1)
print(f"\nSTV (vote transférable) — gagnant : {gagnants_stv[0]}")
print(f"  Tours d'élimination : {len(tours)}")

# Copeland : victoires pairwise − défaites.
g_copeland, scores_copeland = sc.copeland(ballots, OPTIONS)
print(f"\nCopeland — gagnant : {g_copeland}")
print("  Scores (victoires − défaites) :",
      {o: scores_copeland[o] for o in OPTIONS})

Approval voting (chacun approuve son top-2) :
  Burger    5 approbations
  Sushi     4 approbations
  Pizza     3 approbations
  Raclette  2 approbations
→ Gagnant : Burger

STV (vote transférable) — gagnant : Burger
  Tours d'élimination : 3

Copeland — gagnant : Burger
  Scores (victoires − défaites) : {'Pizza': -3, 'Burger': 3, 'Sushi': 1, 'Raclette': -1}


## §4 — Le paradoxe de Condorcet : préférences cycliques

Condorcet a aussi montré que des préférences individuelles *parfaitement rationnelles* peuvent
produire une **cycle collective** : A bat B, B bat C, C bat A. Il n'existe alors **aucun** gagnant
de Condorcet. Construisons un tel profil à 3 électeurs.

In [7]:
# Profil cyclique (3 électeurs, 3 options) — synthétique.
OPTIONS_CYCLE = ["A", "B", "C"]
PROFIL_CYCLE = [
    ("Électeur 1", "stubborn", ["A", "B", "C"]),
    ("Électeur 2", "stubborn", ["B", "C", "A"]),
    ("Électeur 3", "stubborn", ["C", "A", "B"]),
]
ballots_cycle = [p for _, _, p in PROFIL_CYCLE]

print("Duels pairwise (profil cyclique) :")
for o1 in OPTIONS_CYCLE:
    for o2 in OPTIONS_CYCLE:
        if o1 < o2:
            pour = sum(1 for p in ballots_cycle if p.index(o1) < p.index(o2))
            vainqueur = o1 if pour > len(ballots_cycle) / 2 else o2
            print(f"  {o1} vs {o2} : {pour}-{len(ballots_cycle)-pour}  →  {vainqueur} gagne")

gc = sc.condorcet_winner(ballots_cycle, OPTIONS_CYCLE)
print(f"\nGagnant de Condorcet : {gc}")
print("→ Aucun : cycle A > B > C > A. C'est le paradoxe de Condorcet.")

Duels pairwise (profil cyclique) :
  A vs B : 2-1  →  A gagne
  A vs C : 1-2  →  C gagne
  B vs C : 2-1  →  B gagne

Gagnant de Condorcet : None
→ Aucun : cycle A > B > C > A. C'est le paradoxe de Condorcet.


**Leçon n°2.** Quand les préférences cyclent, *aucune* méthode ne peut prétendre élire « le »
vainqueur légitime — il n'en existe pas. Les méthodes divergent alors par construction (Borda,
Copeland, Schulze peuvent chacune désigner un candidat différent). C'est précisément la faille
que formalise le théorème d'Arrow : l'agrégation parfaite est impossible ; reste à choisir une
méthode et à en assumer les arbitrages.

## §5 — Vote stratégique : la Borda est manipulable

Le théorème de Gibbard-Satterthwaite (1973) étend Arrow au vote *stratégique* : toute méthode
non dictatoriale à au moins 3 options est **manipulable** — un électeur a intérêt à ne pas voter
ses vraies préférences. Illustrons-le sur la Borda.

Rappel : au §2, le bloc Pizza (Alice, Bob, Chloé) obtient Pizza = 9 et Burger = 13 (Burger gagne).
Tactique naïve : *enterrer* le rival Burger en le classant dernier. Voyons l'effet.

In [8]:
# Le bloc Pizza « enterre » Burger (le descend de 2e à dernier).
PROFIL_MANIP = [
    ("Alice",  "stubborn", ["Pizza",   "Sushi",   "Raclette", "Burger"]),   # Burger enterré
    ("Bob",    "stubborn", ["Pizza",   "Sushi",   "Raclette", "Burger"]),
    ("Chloé",  "stubborn", ["Pizza",   "Sushi",   "Raclette", "Burger"]),
    ("David",  "flexible", ["Raclette", "Sushi", "Burger",   "Pizza"]),
    ("Emma",   "flexible", ["Raclette", "Sushi", "Burger",   "Pizza"]),
    ("Farid",  "flexible", ["Sushi",   "Burger",  "Raclette","Pizza"]),
    ("Gérald", "flexible", ["Burger",  "Sushi",   "Raclette","Pizza"]),
]

scores_avant = {"Pizza": 9, "Burger": 13, "Sushi": 12, "Raclette": 8}
scores_apres = {o: 0 for o in OPTIONS}
for _, _, prefs in PROFIL_MANIP:
    for i, o in enumerate(prefs):
        scores_apres[o] += n - i - 1

print(f"{'Option':<10} {'Avant':>6} {'Après':>6}")
print("-" * 26)
for o in OPTIONS:
    print(f"{o:<10} {scores_avant[o]:>6} {scores_apres[o]:>6}")
print(f"\nGagnant Borda AVANT manipulation : {max(scores_avant, key=scores_avant.get)}")
print(f"Gagnant Borda APRÈS manipulation : {max(scores_apres, key=scores_apres.get)}")

Option      Avant  Après
--------------------------
Pizza           9      9
Burger         13      7
Sushi          12     15
Raclette        8     11

Gagnant Borda AVANT manipulation : Burger
Gagnant Borda APRÈS manipulation : Sushi


**Leçon n°3 — la manipulation peut se retourner contre son auteur.** En enterrant Burger, le bloc
Pizza espérait faire gagner Pizza. Résultat : c'est le **Sushi** qui l'emporte — un candidat que
le bloc Pizza classe *troisième*. La tactique a **backfire**. C'est la pathologie fameuse de la
Borda : elle récompense le vote tactique, mais ce vote est instable et risqué. Un système de
gouvernance multi-agents robuste doit donc soit choisir une méthode moins manipulable, soit
modéliser explicitement l'incitation stratégique.

## Conclusion

| Leçon | Ce qu'on a prouvé firsthand |
|-------|------------------------------|
| La méthode décide du gagnant | Pluralité = Pizza (perdant de Condorcet) ; Borda/Condorcet = Burger (gagnant de Condorcet) |
| Les préférences peuvent cycler | Profil à 3 électeurs : cycle A>B>C>A, aucun gagnant de Condorcet |
| Toute méthode est manipulable | Enterrer Burger fait élire Sushi (backfire) sous Borda |

L'axe gouvernance du système d'analyse argumentative fournit **7 méthodes** de vote plus les
procédures classiques du choix social, toutes exécutables sur le même profil. En IA multi-agents,
le choix de la méthode de gouvernance est une décision de *conception* à expliciter — jamais un
détail neutre.

### Références
- Condorcet, *Essai sur l'application de l'analyse à la probabilité des décisions rendues à la pluralité des voix* (1785).
- Arrow, *Social Choice and Individual Values* (1951) — théorème d'impossibilité.
- Gibbard (1973) / Satterthwaite (1975) — toute règle non dictatoriale à ≥3 options est manipulable.

---
*Note d'essence* : ce notebook importe `argumentation_analysis.agents.core.governance` depuis le code local.
Lors du refresh du pin essence CoursIA (`coursia-essence-*`), les modules `governance_methods`,
`social_choice` et `governance_agent` devront être importés depuis l'`argumentation_lib/` vendoré —
surface API identique. Voir `docs/architecture/COURSIA_ESSENCE_EXPORT.md`.